# Ablation Comparison

Compare recovery and hallucination rate across ablation variants.

In [1]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

In [2]:
# ── Config ────────────────────────────────────────────────────────────────────
DATASET  = 'pond_ten'
# Leave entries as None unless there is a specific date you need
if DATASET == 'pond_ten':
    ABLATIONS = {
        'llama-3.1-8b': {'baseline': '2026_04_22', '1': '2026_04_20', '2': '2026_04_20', '3': '2026_04_20', '4': '2026_04_21', '5': '2026_04_21', '6': '2026_04_21'},
        'gemma-3-27b': {'baseline': '2026_04_14', '1': '2026_04_15', '2': '2026_04_15', '3': '2026_04_15', '4': '2026_04_15', '5': '2026_04_16', '6': '2026_04_20'},
        'gpt-oss-120b': {'baseline': '2026_04_14', '1': '2026_04_14', '2': '2026_04_21', '3': '2026_04_21', '4': '2026_04_21', '5': '2026_04_21', '6': '2026_04_21'},
    }
elif DATASET == 'nfix_ten':
    ABLATIONS = {
        'llama-3.1-8b': {'baseline': '2026_04_22', '1': '2026_04_21', '2': '2026_04_22', '3': '2026_04_22', '4': '2026_04_22', '5': '2026_04_22', '6': '2026_04_22'},
        'gemma-3-27b': {'baseline': '2026_04_14', '1': '2026_04_16', '2': '2026_04_16', '3': '2026_04_16', '4': '2026_04_16', '5': '2026_04_16', '6': '2026_04_20'},
        'gpt-oss-120b': {'baseline': '2026_04_14', '1': '2026_04_22', '2': '2026_04_22', '3': '2026_04_22', '4': None, '5': None, '6': None},
    }

In [ ]:
import re
import pandas as pd
from analysis.loaders import load_extraction, load_ablation, load_combined_judgements, load_ground_truth
from analysis.metrics import recovery_rate, hallucination_rate
from analysis.plots import recovery_bar
from run_extraction import load_dataset_config
from scholarlm.utils.unit_conversion import apply_unit_conversion
import paths

config = load_dataset_config(DATASET)
ground_truth_df = load_ground_truth(config)


# Old nfix_clean_unit (commented out — too narrow: only handled (mol|g) + (N|C),
# missed N2, C2H4, new-notation superscripts, µ symbol, etc.)
#
# SUPERSCRIPT_MAP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
# SUBSCRIPT_MAP   = str.maketrans("₀₁₂₃₄₅₆₇₈₉₋₊", "0123456789-+")
# SCRIPT_MAP = {**SUPERSCRIPT_MAP, **SUBSCRIPT_MAP}
# LATEX_SCRIPT_RE = re.compile(r"[\^_]\{([^}]*)\}|[\^_]([+-]?\d+)")
# ELEMENT_DASH_RE = re.compile(r"(mol|g) (N|C)")
# TIME_UNIT_RE = re.compile(r"\bday\b|\bhr\b")
# TIME_UNIT_SUB = {"day": "d", "hr": "h"}
# def nfix_clean_unit(s: str) -> str:
#     if not isinstance(s, str):
#         return s
#     s = s.translate(SCRIPT_MAP)
#     s = LATEX_SCRIPT_RE.sub(lambda m: m.group(1) if m.group(1) is not None else m.group(2), s)
#     s = ELEMENT_DASH_RE.sub(r"\1-\2", s)
#     s = TIME_UNIT_RE.sub(lambda m: TIME_UNIT_SUB[m.group()], s)
#     return s

_SUPER_MAP  = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_SUB_MAP    = str.maketrans("₀₁₂₃₄₅₆₇₈₉₋₊", "0123456789-+")
_SCRIPT_MAP = {**_SUPER_MAP, **_SUB_MAP}

# ^{-2} or ^-1 style LaTeX exponents
_LATEX_RE = re.compile(r"[\^_]\{([^}]*)\}|[\^_]([+-]?\d+)")

# "{prefix}[ -]{CHEM}" → "{prefix}-{CHEM}"  where CHEM starts with an uppercase letter
# Handles both old dash notation (nmol-N) and new space notation (nmol N),
# and any chemical: N, N2, C2H4, etc.
_COMPOUND_RE = re.compile(r"(\w+)[\s\-]([A-Z][a-zA-Z0-9]*)")


def nfix_clean_unit(s: str) -> str:
    """Normalise an nfix unit string to the ground-truth format.

    Target: lowercase, hyphen-compound (nmol-n), ASCII exponents (-1/-2/-3),
    u for micro, y for year.

    Handles: new superscript notation (nmol N g⁻¹ h⁻¹), old dash notation
    (nmol-N g-1 h-1), LaTeX (^{-2}), µ symbol, yr vs y, day/hr variants.
    """
    if not isinstance(s, str):
        return s
    # 1. Unicode super/subscripts → ASCII
    s = s.translate(_SCRIPT_MAP)
    # 2. LaTeX-style exponents → ASCII  (^{-2} → -2, ^-1 → -1)
    s = _LATEX_RE.sub(lambda m: m.group(1) if m.group(1) is not None else m.group(2), s)
    # 3. Micro symbol → u
    s = s.replace('µ', 'u').replace('μ', 'u')
    # 4. Compound separator: normalise to hyphen
    #    "nmol N"  → "nmol-N",  "nmol-N"  → "nmol-N" (no-op),
    #    "nmol N2" → "nmol-N2", "ug N"    → "ug-N"
    s = _COMPOUND_RE.sub(r"\1-\2", s)
    # 5. yr → y  (before lowercasing; \b works because - is a non-word char)
    s = re.sub(r'\byr\b', 'y', s)
    # 6. Lowercase everything (L→l, N→n, C→c, etc.)
    s = s.lower()
    # 7. Remaining time-unit variants
    s = re.sub(r'\bday\b', 'd', s)
    s = re.sub(r'\bhr\b',  'h', s)
    return s

if 'pond' in DATASET:
    STRICT_MATCHING = {'document_id': 'document_id', 'attribute': 'attribute', 'value': 'converted_value', 'units': 'units'}
    FUZZY_MATCHING  = {'name': 'name', 'location': 'location', 'ecosystem': 'ecosystem'}
elif 'nfix' in DATASET:
    STRICT_MATCHING = {'document_id': 'document_id', 'attribute': 'attribute', 'value': 'converted_value'}
    FUZZY_MATCHING  = {"name": "name", "site_type": "site_type"}
else:
    raise ValueError("Dataset not recognized.")

## Load all ablations

In [4]:
rows = []

for model in ABLATIONS:
    row = {'model': model}
    
    # Baseline
    baseline_date = ABLATIONS[model]['baseline']
    baseline_path = paths.find_extraction_final(DATASET, model, baseline_date)
    baseline_date = Path(baseline_path).parent.name
    baseline_records = load_extraction(DATASET, model, baseline_date)
    baseline_df = pd.DataFrame(baseline_records)
    baseline_df = apply_unit_conversion(baseline_df, {})
    
    if 'nfix' in DATASET:
        baseline_df['attribute'] = baseline_df['attribute'].map({'nfix_rate_areal': 'nfix_rate', 'nfix_rate_volumetric': 'nfix_rate', 'nfix_rate_mass': 'nfix_rate', 'nfix_rate': 'nfix_rate'})
        baseline_df["units"] = baseline_df["units"].apply(nfix_clean_unit)

    baseline_cache_path = paths.extraction(DATASET, model, baseline_date) / 'match_cache.pkl'
    baseline_recov = recovery_rate(ground_truth_df, baseline_df, strict_matching=STRICT_MATCHING, fuzzy_matching=FUZZY_MATCHING, cache_path = baseline_cache_path)
    baseline_hall = hallucination_rate(ground_truth_df, baseline_df, strict_matching=STRICT_MATCHING, fuzzy_matching=FUZZY_MATCHING, cache_path = baseline_cache_path)
    row['baseline_recov'] = baseline_recov
    row['baseline_hall'] = baseline_hall
    
    for ablation_n in ABLATIONS[model]:
        if ablation_n == 'baseline':
            continue
        try:
            ablation_n_date = ABLATIONS[model][ablation_n]
            ablation_n_path = paths.find_extraction_final(DATASET, model, ablation_n_date, ablation_n)
            ablation_n_date = Path(ablation_n_path).parent.name
            records = load_ablation(DATASET, ablation_n, model, ablation_n_date)
            if len(records) == 0:
                row[f'ablation_{ablation_n}_recov'] = None
                row[f'ablation_{ablation_n}_hall'] = None
                continue
                
            df = pd.DataFrame(records)
            df = apply_unit_conversion(df, {})
    
            if 'nfix' in DATASET:
                df['attribute'] = df['attribute'].map({'nfix_rate_areal': 'nfix_rate', 'nfix_rate_volumetric': 'nfix_rate', 'nfix_rate_mass': 'nfix_rate', 'nfix_rate': 'nfix_rate'})
                df["units"] = df["units"].apply(nfix_clean_unit)

            ablation_n_cache_path = paths.ablation(DATASET, ablation_n, model, ablation_n_date) / 'match_cache.pkl'
            
            ablation_n_recov = recovery_rate(ground_truth_df, df, strict_matching=STRICT_MATCHING, fuzzy_matching=FUZZY_MATCHING, cache_path = ablation_n_cache_path)
            ablation_n_hall = hallucination_rate(ground_truth_df, df, strict_matching=STRICT_MATCHING, fuzzy_matching=FUZZY_MATCHING, cache_path = ablation_n_cache_path)
            row[f'ablation_{ablation_n}_recov'] = ablation_n_recov
            row[f'ablation_{ablation_n}_hall'] = ablation_n_hall
        except FileNotFoundError:
            print(f'Ablation {ablation_n}: no results found, skipping.')
            
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('model')
summary_df.round(3)

,baseline_recov,baseline_hall,ablation_1_recov,ablation_1_hall,ablation_2_recov,ablation_2_hall,ablation_3_recov,ablation_3_hall,ablation_4_recov,ablation_4_hall,ablation_5_recov,ablation_5_hall,ablation_6_recov,ablation_6_hall
model,,,,,,,,,,,,,,
llama-3.1-8b,0.493,0.787,0.566,0.617,0.602,0.656,0.502,0.833,0.277,0.621,NaN,NaN,0.566,0.747
gemma-3-27b,0.723,0.455,0.176,0.492,0.738,0.338,0.395,0.423,0.402,0.274,NaN,NaN,0.563,0.232
gpt-oss-120b,0.823,0.203,0.793,0.211,0.839,0.191,0.837,0.206,0.746,0.180,0.808,0.226,0.867,0.188
